# pygym: browser-notebook check
Open this in the browser notebook you want to test (JupyterLab, classic Notebook, **Google Colab**, Kaggle, VS Code web). Run the cells top to bottom and note, for each, whether what you SEE matches the **Expect** line.

Needs pygym importable in the kernel (`%pip install -e .` from the repo, or `%pip install pygym_cityu`). Not supported: **JupyterLite / Pyodide** (no subprocess or threads in the browser; pygym needs a real Python kernel).

In [ ]:
# 0. Which pygym is this? (both builds say 0.2.0, so the version number cannot tell)
import pygym, inspect
from pygym import jupyter, style
print('imported from:', pygym.__file__)
patched = 'Colab' in (jupyter.in_notebook.__doc__ or '')
print('has the Colab fix:', patched, '| has the dark-notebook code-box fix:', hasattr(style, '_Code'))
# Expect: True / True. If False: this session still has the OLD pygym. Install the new one, then
# Runtime > Restart session (an already-imported pygym is not replaced by a new install):
#   from google.colab import files; files.upload()      # choose pygym-JC.zip
#   !unzip -q -o pygym-JC.zip && pip install -q -e pygym-JC

In [ ]:
# 1. Where are we?
import sys
from IPython import get_ipython
from pygym import jupyter
ip = get_ipython()
print('python        ', sys.version.split()[0])
print('shell class   ', type(ip).__module__ + '.' + type(ip).__name__)
print('class chain   ', [c.__name__ for c in type(ip).__mro__][:4])
print('in_notebook() ', jupyter.in_notebook())
# Expect: in_notebook() True. On Colab the class is google.colab._shell.Shell (a subclass of
# ZMQInteractiveShell). Before the fix this printed False there.

In [ ]:
# 2. Which front-end does exercise() choose?
from pygym import ui
print('auto front-end:', ui._auto_ui())
# Expect: jupyter

In [ ]:
# 3. The progress line. Expect ONE line that animates in place for ~8 s, then DISAPPEARS.
#    (not a new line every tick, no flicker of the whole cell output)
import time
class DemoJob:
    def __init__(self, seconds): self.t0 = time.monotonic(); self.seconds = seconds; self.cancelled = False
    @property
    def elapsed(self): return time.monotonic() - self.t0
    @property
    def progress(self):
        return {'attempt': 1 + int(self.elapsed // 3), 'max_attempts': 4, 'elapsed': self.elapsed,
                'budget': float(self.seconds), 'stage': 'asking', 'detail': ''}
    def wait(self, timeout=None):
        time.sleep(timeout or 0.25); return self.elapsed >= self.seconds
    def cancel(self): self.cancelled = True
job = DemoJob(8)
jupyter._wait(job)
print('finished; cancelled =', job.cancelled)

In [ ]:
# 4. Interrupt. Run this cell and press the STOP / ■ button after ~2 s.
#    Expect: it stops within a second, NO traceback, the progress line vanishes, and it prints cancelled = True.
job = DemoJob(60)
jupyter._wait(job)
print('cancelled =', job.cancelled)

In [ ]:
# 5. The real thing (preset exercise, no AI needed).
from pygym import exercise
exercise('zip')
# Expect: the question panel, then a line 'Your answer goes in a NEW cell below this one'.
# There is no input box in this cell's output: the answer goes in the NEXT cell (cell 6).
# The code box must be dark, NOT a white box with pale text.

**Cell 6 is your answer cell.** Expect `tests 3/3 passed` and `Passed!`. Change `dict(zip(keys, values))` to `{}` and run it again: you should see a red failing row instead.

In [ ]:
%%pygym :w
def combine(keys, values):
    return dict(zip(keys, values))

## What to send back
For every numbered cell: what you saw, plus the browser, the notebook product (JupyterLab / Colab / ...), and the OS. A screenshot of cell 3 while it runs is the most useful thing.